# 🔴 Hard: DMD2 Distribution Matching Loss

Implement the core loss of **DMD2** (Improved Distribution Matching Distillation), the objective that
distills a 50-step diffusion teacher into a **one-step** generator.

### Core Idea

We want a one-step generator $G$ whose output distribution $p_\text{fake}$ matches the teacher's
$p_\text{real}$. Minimise the reverse KL:

$$\mathcal{L} = D_{KL}(p_\text{fake} \,\|\, p_\text{real})$$

You cannot evaluate either density — but you never need to. Its gradient w.r.t. a *generated sample*
is exactly the **difference of two scores**:

$$\nabla_x D_{KL} = \underbrace{\nabla_x \log p_\text{fake}(x)}_{s_\text{fake}} - \underbrace{\nabla_x \log p_\text{real}(x)}_{s_\text{real}}$$

and diffusion models *are* score estimators. So you keep two denoisers around:

| | what it is | how it is trained |
|---|---|---|
| $\mu_\text{real}$ | the frozen teacher | not trained |
| $\mu_\text{fake}$ | an online copy | ordinary diffusion loss on the generator's own outputs |

In $x_0$-prediction space the score difference is proportional to
$(\mu_\text{fake} - \mu_\text{real})$, giving the **distribution matching gradient**:

$$\text{grad} = \frac{\mu_\text{fake}(x) - \mu_\text{real}(x)}{\underbrace{\text{mean}\,|\mu_\text{real}(x) - x|}_{\text{per-sample normalizer}}}$$

The normalizer makes the update scale-free, which is what lets a single learning rate work across
noise levels and resolutions.

**The trick that makes it trainable:** autograd wants a *loss*, not a gradient. So build a surrogate
whose derivative is the gradient you already computed:

$$\mathcal{L} = \tfrac{1}{2}\,\text{MSE}\big(x,\ \text{stopgrad}(x - \text{grad})\big) \quad\Longrightarrow\quad \frac{\partial \mathcal{L}}{\partial x} = \frac{\text{grad}}{N}$$

The `stopgrad` matters: the teacher is frozen, and the fake denoiser is trained by its *own* loss, so
no gradient may flow into either prediction from here.

**What "2" adds to DMD:** (1) drops DMD's expensive regression loss and the teacher-ODE dataset it
required, (2) a **two time-scale update** — refresh $\mu_\text{fake}$ several times per generator
step so the fake score never goes stale, (3) an extra **GAN loss** on real data, which lets the
student *surpass* its teacher, (4) a multi-step (4-step) generator with backward simulation.
This exercise implements the piece all of that is built on.

### Signature
```python
def dmd_loss(x_gen, pred_real, pred_fake, eps=1e-8):
    # x_gen:     generator output, shape (B, ...), requires grad
    # pred_real: frozen teacher's x0-prediction, same shape
    # pred_fake: online fake-score model's x0-prediction, same shape
    # returns: scalar loss whose gradient w.r.t. x_gen is grad / numel
    ...
```

### Rules
- The normalizer is computed **per sample** (mean over every non-batch dim, `keepdim=True`)
- The regression target must be **detached** — no gradient into `pred_real` / `pred_fake`
- Guard the division with `eps` and `torch.nan_to_num`

### Example
```
x_gen     = torch.randn(4, 3, 8, 8, requires_grad=True)
pred_real = torch.randn(4, 3, 8, 8)
pred_fake = torch.randn(4, 3, 8, 8)
loss = dmd_loss(x_gen, pred_real, pred_fake)
loss.backward()
# x_gen.grad == (pred_fake - pred_real) / normalizer / x_gen.numel()
```

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def dmd_loss(x_gen, pred_real, pred_fake, eps=1e-8):
    # x_gen: generator output (B, ...)
    # pred_real / pred_fake: x0-predictions of the frozen teacher / online fake score model
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
x = torch.randn(4, 3, 8, 8, requires_grad=True)
pred_real = torch.randn(4, 3, 8, 8)
pred_fake = torch.randn(4, 3, 8, 8)

loss = dmd_loss(x, pred_real, pred_fake)
loss.backward()
print("loss           :", loss.item())
print("grad shape     :", tuple(x.grad.shape))

normalizer = (pred_real - x.detach()).abs().mean(dim=(1, 2, 3), keepdim=True)
expected = (pred_fake - pred_real) / (normalizer + 1e-8) / x.numel()
print("matches theory :", torch.allclose(x.grad, expected, atol=1e-6))
print("zero when equal:", dmd_loss(x, pred_real, pred_real).item())

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("dmd2")